# 6. Classical baselines (LEAR)

The **LEAR** (LASSO Estimated AutoRegressive) model is the workhorse of
day-ahead electricity price forecasting. The idea is simple but powerful:
fit one LASSO regression *per delivery half-hour*, so the model can learn
which features matter at each time of day.

This notebook trains, evaluates, and dispatches LEAR forecasts on the
SA1 region, building the classical benchmark that later ML models must beat.

## Objectives

- Understand why per-hour modelling outperforms a single global regression.
- Visualise the LASSO regularisation path and feature selection heatmap.
- Train LEAR on the fixed training window, score on the held-out test period.
- Compare per-hour vs global, rolling refit vs fit-once.
- Score: MAE, relative MAE vs similar-day naive.
- Run LEAR forecasts through battery dispatch MPC; report capture ratio.

## Prerequisites

- Notebook 05: feature matrix and backtest harness.
- Processed parquet file `data/processed/SA1_30min.parquet`.

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from grian.config import load_config, repo_root
from grian.dispatch import capture_ratio, schedule
from grian.features import build_matrix
from grian.metrics import mae, relative_mae
from grian.models.baselines import similar_day_naive
from grian.models.lear import LEAR
from grian.viz import apply_style, save_fig

warnings.filterwarnings("ignore", category=UserWarning)

cfg = load_config()
apply_style()

SEED = cfg["seed"]
np.random.seed(SEED)

---
## 1. The LEAR idea

Most time-series models fit a single equation across all hours. But electricity
prices are driven by *different* fundamentals at different times:

- **4 am**: base-load demand, wind generation dominates.
- **1 pm**: solar floods the market, depressing prices.
- **7 pm**: evening ramp -- gas peakers set the price, solar collapses.

LEAR handles this by fitting **48 independent LASSO regressions**, one per
30-minute delivery slot. Each sub-model discovers its own relevant features
through L1 regularisation (LASSO), automatically zeroing out irrelevant inputs.

The LASSO penalty `alpha` is chosen by cross-validation within each slot.

### Load the 30-minute dataset

In [ ]:
# Load 30-min dataset
df = pd.read_parquet(repo_root() / "data" / "processed" / "SA1_30min.parquet")
print(f"Shape: {df.shape}")
print(f"Date range: {df.index.min()} to {df.index.max()}")
df.head()

In [ ]:
df.describe()

### Build feature matrix and target

In [ ]:
# Build feature matrix from price and demand columns
X = build_matrix(df[["price"]], df[["demand"]])
y = np.arcsinh(df["price"]).reindex(X.index)

print(f"Feature matrix shape: {X.shape}")
print(f"Features: {list(X.columns)}")
X.head()

In [ ]:
# Split into train and test
train_start = cfg["train_start"]
train_end = cfg["train_end"]
test_start = cfg["test_start"]
test_end = cfg["test_end"]

X_train = X.loc[train_start:train_end]
y_train = y.loc[train_start:train_end]
X_test = X.loc[test_start:test_end]
y_test = y.loc[test_start:test_end]

print(f"Train: {len(X_train):,} rows  ({train_start} to {train_end})")
print(f"Test:  {len(X_test):,} rows  ({test_start} to {test_end})")

---
## 2. Feature selection: the LASSO path

Before fitting the full LEAR, let's see how LASSO works on a single slot.
As the regularisation strength `alpha` increases, more and more coefficients
are driven to zero. The *regularisation path* shows which features survive.

In [ ]:
# Demonstrate the LASSO path for slot 26 (1:00 PM, peak solar)
slot = 26  # 13:00
hh = X_train.index.hour * 2 + X_train.index.minute // 30
mask = hh == slot
X_slot = X_train.loc[mask]
y_slot = y_train.loc[mask]

# Fit LASSO paths over a range of alphas
alphas = np.logspace(-4, 1, 50)
coef_paths = []
for a in alphas:
    from sklearn.linear_model import Lasso
    lasso = Lasso(alpha=a, max_iter=5000, random_state=SEED)
    lasso.fit(X_slot.values, y_slot.values)
    coef_paths.append(lasso.coef_)

coef_paths = np.array(coef_paths)  # shape: (n_alphas, n_features)

fig, ax = plt.subplots(figsize=(12, 5))
for j, fname in enumerate(X.columns):
    ax.plot(alphas, coef_paths[:, j], label=fname)
ax.set_xscale("log")
ax.set_xlabel("alpha (regularisation strength)")
ax.set_ylabel("Coefficient value")
ax.set_title(f"LASSO regularisation path -- slot {slot} (13:00)")
ax.legend(loc="upper right", fontsize=9)
ax.axhline(0, color="grey", linewidth=0.5)
fig.tight_layout()
save_fig(fig, "lear_lasso_path_slot26")
plt.show()

As alpha increases (moving right), features are progressively zeroed out.
The last features standing are the strongest price drivers for that hour.

### Feature selection heatmap across all 48 slots

Now let's fit the full per-hour LEAR and visualise which features are selected
(non-zero coefficient) at each half-hour.

In [ ]:
# Fit full per-hour LEAR on training data
lear = LEAR(per_hour=True, seed=SEED)
lear.fit(X_train, y_train)

print(f"Fitted {len(lear.models_)} slot models")

In [ ]:
# Coefficient matrix: rows = slots, columns = features
coef_df = lear.coef_matrix()

# Heatmap of feature selection (non-zero = selected)
selection_mask = (coef_df != 0).astype(int)

fig, ax = plt.subplots(figsize=(12, 8))
im = ax.imshow(selection_mask.values, aspect="auto", cmap="Blues",
               interpolation="nearest")
ax.set_xlabel("Feature")
ax.set_ylabel("Half-hour slot")
ax.set_title("Feature selection by half-hour slot (blue = selected by LASSO)")
ax.set_xticks(range(len(coef_df.columns)))
ax.set_xticklabels(coef_df.columns, rotation=45, ha="right", fontsize=9)
ax.set_yticks(range(0, 48, 4))
ax.set_yticklabels([f"{s // 2:02d}:{(s % 2) * 30:02d}" for s in range(0, 48, 4)])
fig.tight_layout()
save_fig(fig, "lear_feature_selection_heatmap")
plt.show()

In [ ]:
# Coefficient magnitude heatmap
fig, ax = plt.subplots(figsize=(12, 8))
vmax = np.percentile(np.abs(coef_df.values), 95)
im = ax.imshow(coef_df.values, aspect="auto", cmap="RdBu_r",
               vmin=-vmax, vmax=vmax, interpolation="nearest")
ax.set_xlabel("Feature")
ax.set_ylabel("Half-hour slot")
ax.set_title("LEAR coefficient magnitudes by slot")
ax.set_xticks(range(len(coef_df.columns)))
ax.set_xticklabels(coef_df.columns, rotation=45, ha="right", fontsize=9)
ax.set_yticks(range(0, 48, 4))
ax.set_yticklabels([f"{s // 2:02d}:{(s % 2) * 30:02d}" for s in range(0, 48, 4)])
plt.colorbar(im, ax=ax, label="Coefficient")
fig.tight_layout()
save_fig(fig, "lear_coef_heatmap")
plt.show()

In [ ]:
# Summary: how many features survive at each slot?
selected = lear.selected_features()
n_selected = {slot: len(feats) for slot, feats in selected.items()}

fig, ax = plt.subplots(figsize=(12, 4))
slots = sorted(n_selected.keys())
ax.bar(slots, [n_selected[s] for s in slots], color="steelblue")
ax.set_xlabel("Half-hour slot")
ax.set_ylabel("Number of selected features")
ax.set_title("Features surviving LASSO selection by slot")
ax.set_xticks(range(0, 48, 4))
ax.set_xticklabels([f"{s // 2:02d}:{(s % 2) * 30:02d}" for s in range(0, 48, 4)])
fig.tight_layout()
save_fig(fig, "lear_n_selected_features")
plt.show()

---
## 3. Fit and predict on the test period

The LEAR model is already fitted on training data. Now generate predictions
for the full test period and invert the `arcsinh` transform so errors are
in \$/MWh.

In [ ]:
# Predict on test set (in arcsinh space)
y_pred_asinh = lear.predict(X_test)

# Invert to dollar space
y_pred = np.sinh(y_pred_asinh)
y_actual = np.sinh(y_test)

print(f"Test predictions: {len(y_pred):,} periods")
print(f"Predicted price range: ${y_pred.min():.1f} to ${y_pred.max():.1f}")
print(f"Actual price range:    ${y_actual.min():.1f} to ${y_actual.max():.1f}")

In [ ]:
# Plot a 2-week sample: actual vs forecast
sample_start = test_start
sample_end = pd.Timestamp(test_start) + pd.Timedelta(days=14)

mask_sample = (y_actual.index >= sample_start) & (y_actual.index < sample_end)

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(y_actual.loc[mask_sample].index, y_actual.loc[mask_sample].values,
        label="Actual", linewidth=1)
ax.plot(y_pred.loc[mask_sample].index, y_pred.loc[mask_sample].values,
        label="LEAR forecast", linewidth=1, alpha=0.8)
ax.set_xlabel("Date")
ax.set_ylabel("Price ($/MWh)")
ax.set_title("LEAR per-hour forecast vs actual -- first two weeks of test")
ax.legend()
fig.tight_layout()
save_fig(fig, "lear_forecast_sample")
plt.show()

---
## 4. Per-hour vs global model

The LEAR's key design choice is fitting one model per half-hour.
How much does this help compared to a single global LASSO?

In [ ]:
# Fit a global (single) LASSO model
lear_global = LEAR(per_hour=False, seed=SEED)
lear_global.fit(X_train, y_train)

# Predict on test set
y_pred_global_asinh = lear_global.predict(X_test)
y_pred_global = np.sinh(y_pred_global_asinh)

# Compare MAE
mae_per_hour = mae(y_actual.values, y_pred.values)
mae_global = mae(y_actual.values, y_pred_global.values)

print(f"MAE (per-hour LEAR):  ${mae_per_hour:.2f}/MWh")
print(f"MAE (global LEAR):    ${mae_global:.2f}/MWh")
print(f"Per-hour improvement: {(1 - mae_per_hour / mae_global) * 100:.1f}%")

In [ ]:
# Break down MAE by hour-of-day for both models
hours = y_actual.index.hour
mae_by_hour_per = []
mae_by_hour_global = []
for h in range(24):
    mask_h = hours == h
    mae_by_hour_per.append(mae(y_actual.values[mask_h], y_pred.values[mask_h]))
    mae_by_hour_global.append(mae(y_actual.values[mask_h], y_pred_global.values[mask_h]))

fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(24)
width = 0.35
ax.bar(x - width / 2, mae_by_hour_per, width, label="Per-hour LEAR", color="steelblue")
ax.bar(x + width / 2, mae_by_hour_global, width, label="Global LEAR", color="coral")
ax.set_xlabel("Hour of day")
ax.set_ylabel("MAE ($/MWh)")
ax.set_title("MAE by hour: per-hour vs global LEAR")
ax.legend()
ax.set_xticks(x)
fig.tight_layout()
save_fig(fig, "lear_per_hour_vs_global_mae")
plt.show()

The per-hour model typically outperforms the global model across most hours,
especially during volatile periods (morning ramp, evening peak) where
feature relevance shifts. The global model is forced to compromise with
a single set of coefficients that cannot adapt to time-of-day dynamics.

---
## 5. Rolling refit vs fit-once

The LEAR so far was fitted once on the full training window. In practice,
market conditions drift. Does refitting weekly on a rolling window help?

In [ ]:
# Rolling refit: refit every 7 days (336 half-hours) on a 1-year window
refit_interval = 336  # 7 days in 30-min periods
window_size = 365 * 48  # 1 year of 30-min periods

test_idx = X_test.index
all_preds_rolling = pd.Series(dtype=float, name="forecast")

# Walk through the test period, refitting every week
n_refits = 0
for start_pos in range(0, len(test_idx), refit_interval):
    end_pos = min(start_pos + refit_interval, len(test_idx))
    forecast_window = test_idx[start_pos:end_pos]

    # Training window: up to the start of this forecast window
    train_end_ts = forecast_window[0] - pd.Timedelta(minutes=30)
    train_start_ts = train_end_ts - pd.Timedelta(days=365)
    if train_start_ts < X.index.min():
        train_start_ts = X.index.min()

    X_tr = X.loc[train_start_ts:train_end_ts]
    y_tr = y.loc[train_start_ts:train_end_ts]

    # Refit LEAR
    lear_rolling = LEAR(per_hour=True, seed=SEED)
    lear_rolling.fit(X_tr, y_tr)

    # Predict this window
    X_fw = X_test.loc[forecast_window]
    preds_asinh = lear_rolling.predict(X_fw)
    preds = np.sinh(preds_asinh)
    all_preds_rolling = pd.concat([all_preds_rolling, preds])
    n_refits += 1

print(f"Rolling refit: {n_refits} refits over the test period")

# Compare MAE
common_idx = y_actual.index.intersection(all_preds_rolling.index)
mae_fit_once = mae(y_actual.loc[common_idx].values, y_pred.loc[common_idx].values)
mae_rolling = mae(y_actual.loc[common_idx].values, all_preds_rolling.loc[common_idx].values)

print(f"\nMAE (fit once):     ${mae_fit_once:.2f}/MWh")
print(f"MAE (weekly refit): ${mae_rolling:.2f}/MWh")
improvement = (1 - mae_rolling / mae_fit_once) * 100
print(f"Rolling refit {'improvement' if improvement > 0 else 'degradation'}: {abs(improvement):.1f}%")

In [ ]:
# Monthly MAE comparison: fit-once vs rolling refit
monthly_mae_once = []
monthly_mae_roll = []
months = []

for month_start in pd.date_range(test_start, test_end, freq="MS"):
    month_end = month_start + pd.offsets.MonthEnd(1)
    idx = common_idx[(common_idx >= month_start) & (common_idx <= month_end)]
    if len(idx) == 0:
        continue
    monthly_mae_once.append(mae(y_actual.loc[idx].values, y_pred.loc[idx].values))
    monthly_mae_roll.append(mae(y_actual.loc[idx].values, all_preds_rolling.loc[idx].values))
    months.append(month_start.strftime("%Y-%m"))

fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(len(months))
width = 0.35
ax.bar(x - width / 2, monthly_mae_once, width, label="Fit once", color="steelblue")
ax.bar(x + width / 2, monthly_mae_roll, width, label="Weekly refit", color="seagreen")
ax.set_xlabel("Month")
ax.set_ylabel("MAE ($/MWh)")
ax.set_title("Monthly MAE: fit-once vs weekly rolling refit")
ax.set_xticks(x)
ax.set_xticklabels(months, rotation=45, ha="right")
ax.legend()
fig.tight_layout()
save_fig(fig, "lear_fit_once_vs_rolling")
plt.show()

Rolling refit tends to help in periods of structural change (e.g. new
generation entering the market, seasonal shifts). The benefit is
incremental -- the LASSO is already robust due to regularisation -- but
for production use, weekly refitting is standard practice.

---
## 6. Scoring: MAE and relative MAE

We score the per-hour LEAR against the similar-day naive baseline.
A relative MAE below 1.0 means the model beats the baseline.

In [ ]:
# Generate naive baseline forecasts over the test period
naive_preds = pd.Series(dtype=float, name="naive")

test_days = pd.date_range(test_start, test_end, freq="D")
for day in test_days:
    origin = day
    try:
        naive_fc = similar_day_naive(df[["price"]], origin=origin, horizon=48)
        naive_preds = pd.concat([naive_preds, naive_fc])
    except Exception:
        continue

# Align all three series on a common index
common = y_actual.index.intersection(y_pred.index).intersection(naive_preds.index)
y_act_c = y_actual.loc[common].values
y_lear_c = y_pred.loc[common].values
y_naive_c = naive_preds.loc[common].values

print(f"Common periods for scoring: {len(common):,}")

In [ ]:
# Score
mae_lear = mae(y_act_c, y_lear_c)
mae_naive = mae(y_act_c, y_naive_c)
rel_mae = relative_mae(y_act_c, y_lear_c, y_naive_c)

print(f"MAE (LEAR per-hour):    ${mae_lear:.2f}/MWh")
print(f"MAE (similar-day naive): ${mae_naive:.2f}/MWh")
print(f"Relative MAE (LEAR / naive): {rel_mae:.3f}")
print()
if rel_mae < 1.0:
    print(f"LEAR beats naive by {(1 - rel_mae) * 100:.1f}%")
else:
    print(f"LEAR underperforms naive by {(rel_mae - 1) * 100:.1f}%")

In [ ]:
# Error distribution: LEAR vs naive
errors_lear = y_act_c - y_lear_c
errors_naive = y_act_c - y_naive_c

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(errors_lear, bins=100, alpha=0.7, color="steelblue", label="LEAR")
axes[0].hist(errors_naive, bins=100, alpha=0.5, color="coral", label="Naive")
axes[0].set_xlabel("Forecast error ($/MWh)")
axes[0].set_ylabel("Count")
axes[0].set_title("Error distribution")
axes[0].legend()
axes[0].set_xlim(-200, 200)

# MAE by hour of day: LEAR vs naive
hours_c = common.hour
mae_lear_by_h = []
mae_naive_by_h = []
for h in range(24):
    m = hours_c == h
    mae_lear_by_h.append(mae(y_act_c[m], y_lear_c[m]))
    mae_naive_by_h.append(mae(y_act_c[m], y_naive_c[m]))

x = np.arange(24)
axes[1].bar(x - 0.2, mae_lear_by_h, 0.35, label="LEAR", color="steelblue")
axes[1].bar(x + 0.2, mae_naive_by_h, 0.35, label="Naive", color="coral")
axes[1].set_xlabel("Hour of day")
axes[1].set_ylabel("MAE ($/MWh)")
axes[1].set_title("MAE by hour: LEAR vs naive")
axes[1].legend()
axes[1].set_xticks(x)

fig.tight_layout()
save_fig(fig, "lear_error_analysis")
plt.show()

### Spike performance

How does LEAR handle price spikes (> $300/MWh)? These drive battery value.

In [ ]:
spike_thresh = cfg["spike_threshold_aud"]
spike_mask = y_act_c > spike_thresh
n_spikes = spike_mask.sum()

if n_spikes > 0:
    mae_lear_spike = mae(y_act_c[spike_mask], y_lear_c[spike_mask])
    mae_naive_spike = mae(y_act_c[spike_mask], y_naive_c[spike_mask])
    print(f"Spike periods (> ${spike_thresh}/MWh): {n_spikes}")
    print(f"MAE on spikes (LEAR):  ${mae_lear_spike:.0f}/MWh")
    print(f"MAE on spikes (naive): ${mae_naive_spike:.0f}/MWh")
else:
    print(f"No spikes above ${spike_thresh}/MWh in test period.")

---
## 7. Dispatch: LEAR forecast to battery revenue

A forecast is only as good as the money it makes. We run the LEAR
day-ahead forecasts through battery dispatch optimisation and compare
revenue against perfect foresight.

For each day in the test period:
1. Take the 48-period LEAR forecast.
2. Solve the LP to get the optimal charge/discharge schedule.
3. Compute revenue against *actual* prices.
4. Compare to perfect-foresight dispatch.

In [ ]:
battery_kwargs = {
    "power_mw": cfg["battery"]["power_mw"],
    "duration_hours": cfg["battery"]["duration_hours"],
    "efficiency": cfg["battery"]["efficiency_roundtrip"],
    "max_cycles": cfg["battery"]["max_cycles_per_day"],
}
print("Battery parameters:", battery_kwargs)

In [ ]:
# Day-ahead dispatch: LEAR forecast vs perfect foresight
daily_revenue_lear = []
daily_revenue_perfect = []
daily_revenue_naive = []
dispatch_days = []

for day in pd.date_range(test_start, test_end, freq="D"):
    day_start = day
    day_end = day + pd.Timedelta(hours=23, minutes=30)

    # Get actual prices for this day
    actual_day = y_actual.loc[day_start:day_end]
    if len(actual_day) < 48:
        continue
    actual_day = actual_day.iloc[:48]

    # Get LEAR forecast for this day
    lear_day = y_pred.reindex(actual_day.index)
    if lear_day.isna().any():
        continue

    # Get naive forecast for this day
    naive_day = naive_preds.reindex(actual_day.index)

    # Dispatch against LEAR forecast, evaluated on actual prices
    result_lear = schedule(lear_day.values, **battery_kwargs)
    result_perfect = schedule(actual_day.values, **battery_kwargs)

    # Revenue from LEAR dispatch against actual prices
    if result_lear["status"] == "optimal" and result_perfect["status"] == "optimal":
        # Actual revenue: use LEAR's dispatch decisions but actual prices
        net_power = result_lear["discharge"] - result_lear["charge"]
        revenue_lear = float(np.sum(actual_day.values * net_power * 0.5))
        daily_revenue_lear.append(revenue_lear)
        daily_revenue_perfect.append(result_perfect["revenue"])
        dispatch_days.append(day)

        # Also dispatch on naive forecast
        if naive_day is not None and not naive_day.isna().any():
            result_naive = schedule(naive_day.values, **battery_kwargs)
            if result_naive["status"] == "optimal":
                net_power_naive = result_naive["discharge"] - result_naive["charge"]
                revenue_naive = float(np.sum(actual_day.values * net_power_naive * 0.5))
                daily_revenue_naive.append(revenue_naive)
            else:
                daily_revenue_naive.append(0.0)
        else:
            daily_revenue_naive.append(0.0)

print(f"Dispatched {len(dispatch_days)} days")

In [ ]:
# Compute capture ratios
total_lear = sum(daily_revenue_lear)
total_perfect = sum(daily_revenue_perfect)
total_naive = sum(daily_revenue_naive)

cr_lear = capture_ratio(total_lear, total_perfect)
cr_naive = capture_ratio(total_naive, total_perfect)

print(f"Total revenue (LEAR dispatch):     ${total_lear:,.0f}")
print(f"Total revenue (naive dispatch):    ${total_naive:,.0f}")
print(f"Total revenue (perfect foresight): ${total_perfect:,.0f}")
print()
print(f"Capture ratio (LEAR):  {cr_lear:.3f}  ({cr_lear * 100:.1f}%)")
print(f"Capture ratio (naive): {cr_naive:.3f}  ({cr_naive * 100:.1f}%)")

In [ ]:
# Cumulative revenue over test period
cum_lear = np.cumsum(daily_revenue_lear)
cum_perfect = np.cumsum(daily_revenue_perfect)
cum_naive = np.cumsum(daily_revenue_naive)

fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(dispatch_days, cum_perfect, label="Perfect foresight",
        linewidth=1.5, color="grey", linestyle="--")
ax.plot(dispatch_days, cum_lear, label=f"LEAR (CR={cr_lear:.1%})",
        linewidth=1.5, color="steelblue")
ax.plot(dispatch_days, cum_naive, label=f"Naive (CR={cr_naive:.1%})",
        linewidth=1.5, color="coral")
ax.set_xlabel("Date")
ax.set_ylabel("Cumulative revenue ($)")
ax.set_title("Cumulative battery revenue: LEAR vs naive vs perfect foresight")
ax.legend()
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"${x:,.0f}"))
fig.tight_layout()
save_fig(fig, "lear_cumulative_revenue")
plt.show()

In [ ]:
# Daily revenue scatter: LEAR vs perfect foresight
fig, ax = plt.subplots(figsize=(8, 8))
ax.scatter(daily_revenue_perfect, daily_revenue_lear, alpha=0.3,
           s=15, color="steelblue")
lims = [min(min(daily_revenue_perfect), min(daily_revenue_lear)),
        max(max(daily_revenue_perfect), max(daily_revenue_lear))]
ax.plot(lims, lims, "--", color="grey", linewidth=1, label="Perfect = LEAR")
ax.set_xlabel("Perfect foresight revenue ($)")
ax.set_ylabel("LEAR dispatch revenue ($)")
ax.set_title("Daily revenue: LEAR dispatch vs perfect foresight")
ax.legend()
fig.tight_layout()
save_fig(fig, "lear_daily_revenue_scatter")
plt.show()

### Summary table

In [ ]:
# Summary table
summary = pd.DataFrame({
    "Model": ["LEAR (per-hour)", "LEAR (global)", "Similar-day naive"],
    "MAE ($/MWh)": [mae_lear, mae_global, mae_naive],
    "Relative MAE": [rel_mae, mae_global / mae_naive, 1.0],
    "Capture ratio": [f"{cr_lear:.1%}", "--", f"{cr_naive:.1%}"],
    "Total revenue ($)": [f"${total_lear:,.0f}", "--", f"${total_naive:,.0f}"],
})
summary.set_index("Model", inplace=True)
summary

---
## Exercises

### Exercise 1: Feature selection at peak solar vs evening peak

Which features survive LASSO selection at slot 26 (1 pm, peak solar)
vs slot 38 (7 pm, evening peak)? What does this tell you about
price drivers at different times of day?

<details><summary>Hint 1</summary>
Use <code>lear.selected_features()</code> to get a dict mapping slot number
to the list of features with non-zero coefficients.
</details>

<details><summary>Hint 2</summary>
Compare the sets: <code>set(selected[26])</code> vs <code>set(selected[38])</code>.
Which features appear in one but not the other?
</details>

<details><summary>Solution</summary>

```python
selected = lear.selected_features()

solar_slot = 26  # 1:00 PM
evening_slot = 38  # 7:00 PM

print(f"Slot {solar_slot} (13:00) selected features:")
for f in selected.get(solar_slot, []):
    print(f"  - {f}")

print(f"\nSlot {evening_slot} (19:00) selected features:")
for f in selected.get(evening_slot, []):
    print(f"  - {f}")

# Compare
solar_set = set(selected.get(solar_slot, []))
evening_set = set(selected.get(evening_slot, []))

print(f"\nOnly in solar peak: {solar_set - evening_set}")
print(f"Only in evening peak: {evening_set - solar_set}")
print(f"In both: {solar_set & evening_set}")

# Coefficient comparison
coefs = lear.coef_matrix()
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, slot, label in [(axes[0], solar_slot, "13:00 (solar peak)"),
                         (axes[1], evening_slot, "19:00 (evening peak)")]:
    vals = coefs.loc[slot]
    nonzero = vals[vals != 0].sort_values()
    nonzero.plot.barh(ax=ax, color="steelblue")
    ax.set_title(f"Slot {slot} -- {label}")
    ax.set_xlabel("Coefficient")
fig.tight_layout()
plt.show()
```
</details>

In [ ]:
# Your analysis here

### Exercise 2: Residuals vs net load

The LEAR assumes a linear relationship between features and price.
Plot the LEAR residuals against demand (as a proxy for net load).
Is there a nonlinear pattern the LEAR misses?

<details><summary>Hint 1</summary>
Residuals are <code>y_actual - y_pred</code>. Use demand from the test
period aligned to the same index.
</details>

<details><summary>Hint 2</summary>
A scatter plot with a LOWESS or rolling-mean overlay will reveal
any systematic curvature. If residuals fan out at high demand,
that's heteroscedasticity -- the LEAR underestimates uncertainty
during high-demand periods.
</details>

<details><summary>Solution</summary>

```python
residuals = y_actual - y_pred
demand_test = df.loc[y_actual.index, "demand"]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter: residuals vs demand
axes[0].scatter(demand_test.values, residuals.values, alpha=0.05, s=5,
                color="steelblue")
axes[0].axhline(0, color="grey", linewidth=1)
axes[0].set_xlabel("Demand (MW)")
axes[0].set_ylabel("Residual ($/MWh)")
axes[0].set_title("LEAR residuals vs demand")

# Binned mean residual to see systematic pattern
binned = pd.DataFrame({"demand": demand_test.values, "residual": residuals.values})
binned["demand_bin"] = pd.qcut(binned["demand"], 20, duplicates="drop")
bin_stats = binned.groupby("demand_bin")["residual"].agg(["mean", "std"])
bin_centers = [interval.mid for interval in bin_stats.index]

axes[1].plot(bin_centers, bin_stats["mean"], "o-", color="steelblue",
             label="Mean residual")
axes[1].fill_between(bin_centers,
                     bin_stats["mean"] - bin_stats["std"],
                     bin_stats["mean"] + bin_stats["std"],
                     alpha=0.2, color="steelblue", label="+/- 1 std")
axes[1].axhline(0, color="grey", linewidth=1)
axes[1].set_xlabel("Demand (MW)")
axes[1].set_ylabel("Mean residual ($/MWh)")
axes[1].set_title("Binned mean residual vs demand")
axes[1].legend()

fig.tight_layout()
plt.show()

print("If the mean residual curve is not flat, there is a systematic")
print("nonlinear pattern the LEAR misses -- opportunity for ML models.")
```
</details>

In [ ]:
# Your analysis here

### Exercise 3: MAE improvement vs revenue improvement

LEAR's relative MAE tells us how much better the *point forecast* is
compared to naive. The capture ratio tells us how that translates to
*revenue*. Are these proportional? Does a 10% better MAE translate
to 10% more revenue?

<details><summary>Hint 1</summary>
Compare <code>(1 - rel_mae)</code> (the MAE improvement fraction) with
<code>(cr_lear - cr_naive) / cr_naive</code> (the revenue improvement fraction).
</details>

<details><summary>Hint 2</summary>
Think about why they might differ: forecast errors during high-price
periods matter more for revenue (the battery acts on price extremes),
while MAE weights all periods equally.
</details>

<details><summary>Solution</summary>

```python
mae_improvement = (1 - rel_mae) * 100
revenue_improvement = ((cr_lear - cr_naive) / cr_naive) * 100 if cr_naive > 0 else float("inf")

print(f"MAE improvement over naive:     {mae_improvement:.1f}%")
print(f"Revenue improvement over naive:  {revenue_improvement:.1f}%")
print(f"Revenue gain per % MAE gain:     {revenue_improvement / mae_improvement:.2f}")
print()
print("Key insight: these are generally NOT proportional.")
print("Battery revenue depends on forecast quality during price EXTREMES,")
print("not the average error across all periods. A model can have a modest")
print("MAE improvement but a large revenue gain if it better captures spikes")
print("and troughs -- the periods that drive charge/discharge decisions.")
print()
print("This is why capture ratio (not MAE) is the primary metric for")
print("forecast-driven dispatch.")

# Monthly breakdown to illustrate the disconnect
monthly_cr_lear = []
monthly_cr_naive = []
monthly_rmae = []
months_disp = []

for i, day in enumerate(dispatch_days):
    month_key = day.strftime("%Y-%m")
    if month_key not in months_disp:
        months_disp.append(month_key)

# Group daily revenues by month
rev_df = pd.DataFrame({
    "day": dispatch_days,
    "lear": daily_revenue_lear,
    "perfect": daily_revenue_perfect,
    "naive": daily_revenue_naive,
})
rev_df["month"] = rev_df["day"].dt.to_period("M")
monthly = rev_df.groupby("month").sum(numeric_only=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
months_str = [str(m) for m in monthly.index]
x = np.arange(len(months_str))

cr_l = monthly["lear"] / monthly["perfect"]
cr_n = monthly["naive"] / monthly["perfect"]

axes[0].bar(x - 0.2, cr_l.values, 0.35, label="LEAR", color="steelblue")
axes[0].bar(x + 0.2, cr_n.values, 0.35, label="Naive", color="coral")
axes[0].set_xlabel("Month")
axes[0].set_ylabel("Capture ratio")
axes[0].set_title("Monthly capture ratio")
axes[0].set_xticks(x)
axes[0].set_xticklabels(months_str, rotation=45, ha="right")
axes[0].legend()

axes[1].bar(x, ((cr_l.values - cr_n.values) / cr_n.values) * 100,
            color="seagreen")
axes[1].set_xlabel("Month")
axes[1].set_ylabel("Revenue improvement (%)")
axes[1].set_title("LEAR revenue improvement over naive (monthly)")
axes[1].set_xticks(x)
axes[1].set_xticklabels(months_str, rotation=45, ha="right")
axes[1].axhline(0, color="grey", linewidth=0.5)

fig.tight_layout()
plt.show()
```
</details>

In [ ]:
# Your analysis here

---
## Summary

- **LEAR** fits one LASSO per delivery half-hour, letting each slot
  select its own relevant features. This consistently outperforms a
  single global linear model.
- **Feature selection** varies across the day: overnight slots rely on
  lagged prices and demand; midday slots respond to different drivers
  as solar suppresses prices.
- **Rolling refit** provides incremental improvement over fit-once,
  adapting to market drift.
- **Capture ratio** is the bottom line: LEAR translates forecast
  accuracy into battery revenue, but MAE improvement and revenue
  improvement are not proportional -- price extremes drive dispatch
  value.
- The LEAR sets the **linear benchmark** that ML models in notebook 07
  must beat.

---
## Report

In [ ]:
# Write report to outputs/reports/
report_dir = repo_root() / "outputs" / "reports"
report_dir.mkdir(parents=True, exist_ok=True)

report = f"""# Notebook 06: Classical Baselines (LEAR) -- Report

Region: {cfg['region']}
Train:  {train_start} to {train_end}
Test:   {test_start} to {test_end}

## Point forecast performance

| Model             | MAE ($/MWh) | Relative MAE |
|-------------------|-------------|--------------|
| LEAR (per-hour)   | {mae_lear:.2f}       | {rel_mae:.3f}         |
| LEAR (global)     | {mae_global:.2f}      | {mae_global / mae_naive:.3f}         |
| Similar-day naive | {mae_naive:.2f}      | 1.000         |

## Battery dispatch

Battery: {cfg['battery']['power_mw']} MW / {cfg['battery']['duration_hours']} hr,
         {cfg['battery']['efficiency_roundtrip']:.0%} round-trip efficiency,
         {cfg['battery']['max_cycles_per_day']} max cycles/day

| Model  | Total revenue | Capture ratio |
|--------|---------------|---------------|
| LEAR   | ${total_lear:,.0f}    | {cr_lear:.1%}         |
| Naive  | ${total_naive:,.0f}   | {cr_naive:.1%}        |
| Perfect| ${total_perfect:,.0f} | 100.0%        |

## Key findings

- Per-hour LEAR outperforms global LEAR by {(1 - mae_per_hour / mae_global) * 100:.1f}% MAE.
- LEAR beats naive baseline with relative MAE of {rel_mae:.3f}.
- LEAR capture ratio: {cr_lear:.1%} vs naive capture ratio: {cr_naive:.1%}.
- Rolling weekly refit {'improves' if improvement > 0 else 'slightly degrades'} on fit-once by {abs(improvement):.1f}%.
"""

report_path = report_dir / "06_classical_baselines.md"
report_path.write_text(report)
print(f"Report written to {report_path}")